# 07 · Graph Classification

## What this notebook covers
Node classification assigns labels to individual nodes. Graph classification assigns a label to an *entire graph*. This requires a *readout* function that aggregates all node embeddings into a single graph-level representation.

## Readout functions
| Function | Notes |
|---------|-------|
| **Sum** | Sensitive to graph size; theoretically most expressive (GIN) |
| **Mean** | Size-invariant; good for graphs of varying sizes |
| **Max** | Captures extreme features; invariant to size |
| **Attention pooling** | Learnable weighted sum; expensive but powerful |
| **DiffPool** | Hierarchical pooling; learns a soft cluster assignment at each layer |

## The WL isomorphism test
The **Weisfeiler-Lehman (WL) graph isomorphism test** iteratively refines node colour assignments based on neighbour colours. It is the theoretical upper bound on GNN expressiveness: GNNs are at most as powerful as 1-WL.

**GIN** (Graph Isomorphism Network, Xu et al. 2019) achieves this upper bound by proving that:
1. Sum aggregation (not mean or max) is necessary
2. The update function must be injective (an MLP with sufficient capacity achieves this)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

try:
    from torch_geometric.datasets import TUDataset
    from torch_geometric.nn import GINConv, global_add_pool, global_mean_pool
    from torch_geometric.loader import DataLoader
    HAS_PYG = True
except ImportError:
    HAS_PYG = False

In [ ]:
if HAS_PYG:
    # MUTAG: 188 molecular graphs, binary mutagenicity labels
    dataset = TUDataset(root="/tmp/MUTAG", name="MUTAG")
    dataset = dataset.shuffle()
    n_train = int(len(dataset) * 0.8)
    train_ds, test_ds = dataset[:n_train], dataset[n_train:]
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=32)

    print(f"MUTAG: {len(dataset)} graphs  |  {dataset.num_classes} classes  |  avg nodes={np.mean([d.num_nodes for d in dataset]):.1f}")

    class GIN(torch.nn.Module):
        def __init__(self, in_channels, hidden_channels, out_channels):
            super().__init__()
            mlp1 = nn.Sequential(nn.Linear(in_channels, hidden_channels), nn.ReLU(), nn.Linear(hidden_channels, hidden_channels))
            mlp2 = nn.Sequential(nn.Linear(hidden_channels, hidden_channels), nn.ReLU(), nn.Linear(hidden_channels, hidden_channels))
            self.conv1 = GINConv(mlp1)
            self.conv2 = GINConv(mlp2)
            self.lin   = nn.Linear(hidden_channels, out_channels)

        def forward(self, x, edge_index, batch):
            x = F.relu(self.conv1(x, edge_index))
            x = F.relu(self.conv2(x, edge_index))
            x = global_add_pool(x, batch)   # sum readout
            return self.lin(x)

    gin = GIN(dataset.num_features, 64, dataset.num_classes)
    opt = torch.optim.Adam(gin.parameters(), lr=0.01)

    def train_epoch():
        gin.train()
        total_loss = 0
        for batch in train_loader:
            opt.zero_grad()
            out = gin(batch.x.float(), batch.edge_index, batch.batch)
            loss = F.cross_entropy(out, batch.y)
            loss.backward(); opt.step()
            total_loss += loss.item()
        return total_loss / len(train_loader)

    def evaluate(loader):
        gin.eval()
        correct = total = 0
        with torch.no_grad():
            for batch in loader:
                out = gin(batch.x.float(), batch.edge_index, batch.batch)
                correct += (out.argmax(dim=1) == batch.y).sum().item()
                total   += len(batch.y)
        return correct / total

    train_accs, test_accs = [], []
    for epoch in range(100):
        loss = train_epoch()
        train_accs.append(evaluate(train_loader))
        test_accs.append(evaluate(test_loader))

    print(f"GIN on MUTAG: Final test accuracy = {test_accs[-1]:.4f}")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_accs, label="Train"); ax.plot(test_accs, label="Test")
    ax.set_title("GIN on MUTAG"); ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.legend()
    plt.tight_layout()
    plt.savefig(f"{base}/07_graph_classification.png", dpi=120)
    plt.show()
else:
    print("PyTorch Geometric not available. GIN on MUTAG typically achieves ~85-90% accuracy.")
    print("Key: sum readout + MLP update makes GIN as expressive as WL test.")

## Readout comparison
| Readout | Test acc on MUTAG (typical) | Notes |
|---------|----------------------------|-------|
| Sum (GIN) | 85-90% | Theoretically optimal |
| Mean | 80-85% | Size-invariant |
| Max | 78-83% | Misses multi-occurrence structure |

The theoretical result from GIN: sum > max ≥ mean in terms of discriminative power for graph structures that differ in the *count* of a substructure.